# gRPC 스트리밍 STT

## 학습 목표

이 장을 통해 다음을 학습할 수 있습니다

1. gRPC의 기본 개념과 특징을 이해합니다. 

2. 양방향 스트리밍의 원리와 구현 방법을 학습합니다.

3. Protocol Buffer 메시지 구조를 이해합니다.

4. gRPC 클라이언트 구현을 실습합니다

## gRPC 기본 개념

### gRPC란?

gRPC는 Google이 개발한 고선능 Remote Procedure call 프레임워크

- HTTP/2 기반: 단일 연결로 다중 요청 처리
- Protocol Buffer: 효율적인 바이너리 직렬화
- 양방향 스트리밍: 클라이언와 서버가 동시에 데이터 전송 가느아
- 타입 안전성: 강력한 타입 시스템

REST API vs gRPC

REST API:

- HTTP/1.1 기반

- JSON 텍스트 형ㅅ힉

- 요청-응답 모델

- 브라우저에서 직접 호출 가능

gRPC:

- HTTP/2 기반

- Protocol Buffer 바이너리 형식

- 스트리밍 지우너

- 높은 성능

## gRPC 통신 방식

### gRPC는 4가지 통신 방식을 지원합니다. 

1. Unary RPC: 단일 요청 - 응답
2. Server Streaming: 서버가 스트림으로 응답
3. Client Streaming: 클라이언트가 스트림으로 요청
4. Bidirectinonal Streaming: 양방향 스트리밍 

실시간 STT는 Bidriectinal Streaming을 사용합니다.

# Protocol Buffer

## Protocol Buffer 란? 

효율적: JSON보다 작은 크기
빠름: 파싱 속도가 빠름
타입 안전: 강력한 타입 시스템
버전 호환: 필드 추가/삭제 시 호환성 유지 


메시지 정의 
```proto
.proto 파일에 미시지 구조를 정의합니다.

syntax = "proto3"

message RecognitionConfig {
    string 
}

message RecognitionREsult {

}

message StreamingRecognizieRequest {

}

message StreamingRecognizeResponse {

}

service Speech {
    rpc StreamingRecognize(stream STreamingRecognizeRequest)
    return 
}
```


> 메시지 컴파일

.proto 파일을 Python 코드로 컴파일 합니다. 

```python

python -m grpc_tools.protoc \ --python_out=. \ --grpc_python_out=. \ --proto_path=. \ speech.proto

```

양방향 스트리밍

스트리밍의 필요성

실시간 STT에서는 다음과 같은 이유로 스트리밍이 필요합니다

낮은 지연시간

메모리 호율

실시간 피드백

요청 메시지 생성

첫번재 요청은 설정을 포함하고 이후 요청은 오디오 데이터를 포함합니다.


In [ ]:
from typing import AsyncIterator


class Anything:
    @staticmethod
    async def create_request_messages(
        audio_chunks: AsyncIterator[bytes],
        language_code: str,
        interim_results: bool,
    ) -> AsyncIterator[seech_pb2.StreamingRecognizeRequest]:
        """gRPC 요청 메시지를 생성하는 제너레이터"""
        # 첫 번째 요청: 설정 전송
        config = speech_pb2.RecognitionConfig(
            language_code=language_code,
            interim_results=interim_results
        )

        first_request = speech_pb2.StreamingRecognizeRequest(config=config)
        yield first_request

        async for audio_chunk in audio_chunk:
            if audio_chunk and len(audio_chunk) > 0:
                request = speech_pb2.StreamingRecognizeRequest(
                    audio_content=audio_chunk
                )
                yield request

    

## DagloRealtimeSTTClient 구현

### 클래스 구조

```python

class DagloRealtimeSTTClient:
    def __init__(self, timeout: int = 10800):
        self.channel = None
        self.stub = None
        self.server_address = None
        self.api_token = None
        self.timeout = timeout
```

gRPC 연결

SSL/TLS 보안 채널을 생성하고 서버에 연결합니다.

```python

async def connect(self, server_address: str, api_token: str):
    """gRPC 서버에 연결"""
    if not api_token:
        raise ValueError("api_token은 필수입니다.")
    
    self.server_address = server_address
    self.api_token = api_token

    # SSL/TLS 보안 채널 생성
    credentials = grpc.ssl_channel_credentials()
    self.channel = grpc.aio.secure_channel(
        server_adress,
        credentials,
    )

    # gRPC 스텁 생성
    self.stub = speech_pb2_grpc.SpeechStub(self.channel)

    logger.info(f"gRPC 연결 성공: server_address={server_address}")

    

```

## 양방향 스트리밍 수행

### stream_recongnize 메서드가 양방향 스트리밍을 수행합니다.

```python

async def stream_recognize(
    self,
    audio_chunks: AysyncIterator[bytes],
    langue_code: str = "ko-KR",
    interim_results: bool = True,
) -> AsyncIterator[dict[str, str | bool | float]]:
    """ 실시간 STT 스트리밍을 수행합니다. """

    if self.stub is None or self.channel is None:
        raise RuntimeError("gRPC 연결이 설정되지 않았습니다.")
    
    if language_code not in SUPPORTED_LANGUAGE_CODES:
        raise ValueError(f"지원하지 않는 언어 코드: {language_code}")
    
    # API 인증 메타데이터

    metadata = (("authorization", f"Bearer {self.api_token}"),)

    # 요청 제너레이터 생성

    request_iterator = self.Private.create_request_messages(
        audio_chunks,language_code,interim_results
    )

    # 양방향 스트리미이 호출

    response_tream = self.stub.StreamingRecognize(
        request_iterator,
        metadata=metadata,
        timeout=self.timeout,
    )

    # 응답 스트림 처리

    async for response in response_stream:
        if response.result:
            result_dict = {
                "transcript": response.result.transcript,
                "is_final": response.result.is_final,
                "language_code": response.result.language_code,
                "total_duration": response.total_duration,
            }
            yield result_dict

    
    


```
# 서버 주소 정규화
    ## 주소 형식 변환
    ### 다양한 형식의 API URL을 gRPC 서버 주소로 변환합니다.

    ```python

    class Anything:
        @staticmethod
        def normalizer_grpc_server_address(api_url: str | None) -> str:
            """gRPC 서버 주소를 정규화합니다."""
            if not api_url:
                return "apis.daglo.ai:443"
            


    ```

## 스트림 처리

### STTPipeline.process_stream

#### 서비스 레이어에서 STT 스트림을 처리합니다.


```python

class Anything:
    
    @staticmethod
    async def process_stream(
        audio_chunks: AysncIterator[bytes],
        session_id: UUID,
        stt_config: RealtimeSTTConfig,
        api_token: str,
        language_code: str = "ko-KR"
        interim_results: str = "webm",
        timeout: int = 3600
    ) -> AsyncIterator[dict[str, Any]]:
        """실시간 STT 스트림을 처리합니다."""

        if not stt_config:
            raise ServiceError("STT 설정이 제공되지 않았습니다", status_code=400")
        
        if not api_token:
            raise ServiceError("API 토큰이 제공되지 않았습니다.", status_code=400)
        
        if stt_config.provider != "DAGLO"
            raise ServiceError(f"실시간 STT는 DAGLO만 지원합니다. 현재 provider: {stt_config.provider}", status_code=400)

        
        # 서버 주소 정규화
        oiriginal_url = stt_config.api_url or "apis.daglo.. "

        # 생성 및 ㅇ년결

        # 오디오 변환 제네레이터 생성

        converted_chunks = RealtimeSTTSErve.. 

        

```